In [2]:
from transformers import pipeline
import cv2
import os
from PIL import Image
import numpy as np

# Load the pipeline
pipe = pipeline(task="depth-estimation", model="depth-anything/Depth-Anything-V2-Large-hf")

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.48, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
Device set to use cpu


In [ ]:
# Path to your video file
video_path = ".mp4"  # Update with the actual video path
output_video_path = ".mp4"  # Output video file path

# Open the video file
cap = cv2.VideoCapture(video_path)

# Get video properties
fps = int(cap.get(cv2.CAP_PROP_FPS))
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

# Initialize video writer for saving the output
fourcc = cv2.VideoWriter_fourcc(*"mp4v")  # Codec for MP4
out = cv2.VideoWriter(output_video_path, fourcc, fps, (width, height))

frame_idx = 0
while True:
    ret, frame = cap.read()
    if not ret:
        break  # End of video

    # Convert the frame (OpenCV BGR to RGB)
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    # Convert the frame to PIL Image
    frame_pil = Image.fromarray(frame_rgb)

    # Perform depth estimation
    result = pipe(frame_pil)
    depth = result["depth"]

    # Normalize depth for better visualization (convert to 8-bit grayscale)
    depth_normalized = (depth - np.min(depth)) / (np.max(depth) - np.min(depth))  # Normalize to 0-1
    depth_8bit = (depth_normalized * 255).astype(np.uint8)  # Convert to 0-255

    # Convert to BGR for video writing
    depth_colormap = cv2.applyColorMap(depth_8bit, cv2.COLORMAP_VIRIDIS)

    # Write the processed frame to the output video
    out.write(depth_colormap)

    print(f"Processed frame {frame_idx}/{frame_count}")
    frame_idx += 1

cap.release()
out.release()
print(f"Depth video saved to {output_video_path}")